# Agência Nacional das Águas

Definir a variável de ambiente `RESTAPI_USE_ARCPY` como `FALSE` é necessário para evitar que a biblioteca `restapi` mande mensagens de erro ou tente usar o `ArcPy`, que só está disponível para quem tem licença da ESRI.


In [1]:
import os

os.environ['RESTAPI_USE_ARCPY'] = 'FALSE'

In [2]:
import pprint
import tempfile
import warnings

import requests
import restapi
from restapi import NAME, SERVICES, TYPE, ArcServer

import open_geodata as geo

<br>


-----

## Diretório

In [3]:
# Crio pasta temporária
temp_path = Path(tempfile.gettempdir()) / 'open_geodata' / 'ana'
temp_path.mkdir(exist_ok=True)
temp_path

WindowsPath('C:/Users/michel/AppData/Local/Temp/open_geodata/ana')

<br>


-----


## ArcGIS


In [4]:
# Connect to restapi.ArcServer instance
ags = restapi.ArcServer(url='https://portal1.snirh.gov.br/server/rest/services')
ags

d:\Codes\GitHub\Personal\my_projects\open-geodata\.venv\Lib\site-packages\restapi\rest_utils.py:42: UserWarning: no request client has been set, using default client
  warnings.warn('no request client has been set, using default client')


<ArcServer: "portal1.snirh.gov.br" ("server")>

In [5]:
for x in ags.list_services():
    print(x, type(x))

https://portal1.snirh.gov.br/server/rest/services/APPs_e_AUR_MIL1/MapServer <class 'str'>
https://portal1.snirh.gov.br/server/rest/services/Chuva_Telemétrica/MapServer <class 'str'>
https://portal1.snirh.gov.br/server/rest/services/Conservação_MIL1/MapServer <class 'str'>
https://portal1.snirh.gov.br/server/rest/services/Cotas_de_Referencia/MapServer <class 'str'>
https://portal1.snirh.gov.br/server/rest/services/Divisao_Estadual/MapServer <class 'str'>
https://portal1.snirh.gov.br/server/rest/services/EstacaoSetorEletrico/MapServer <class 'str'>
https://portal1.snirh.gov.br/server/rest/services/Estacoes_Fluviometricas_RETE/MapServer <class 'str'>
https://portal1.snirh.gov.br/server/rest/services/Estacoes_Hidrologicas/MapServer <class 'str'>
https://portal1.snirh.gov.br/server/rest/services/Estacoes_Hidrotelemetria/MapServer <class 'str'>
https://portal1.snirh.gov.br/server/rest/services/Estado_do_Uso_da_Terra_MIL1/MapServer <class 'str'>
https://portal1.snirh.gov.br/server/rest/servic

d:\Codes\GitHub\Personal\my_projects\open-geodata\.venv\Lib\site-packages\restapi\common_types.py:1355: UserWarning: Authentation Error for folder Utilities: 
{'error': {'code': 499, 'message': 'Token Required', 'details': []}}
  warnings.warn('Authentation Error for folder {}: {}{}'.format(s, os.linesep,  e))


In [6]:
for root, services in ags.walk(ignore_folder_auth=True):
    print(f'Pasta: {root}')
    # print('\n'.join(f'- {item}' for item in services))
    for service in services:
        print(f'- {service}')

    print(f'-' * 60)

Pasta: None
- APPs_e_AUR_MIL1/MapServer
- Chuva_Telemétrica/MapServer
- Conservação_MIL1/MapServer
- Cotas_de_Referencia/MapServer
- Divisao_Estadual/MapServer
- EstacaoSetorEletrico/MapServer
- Estacoes_Fluviometricas_RETE/MapServer
- Estacoes_Hidrologicas/MapServer
- Estacoes_Hidrotelemetria/MapServer
- Estado_do_Uso_da_Terra_MIL1/MapServer
- Estações_Hidrometeorológicas_SNIRH/FeatureServer
- Estações_Hidrometeorológicas_SNIRH/MapServer
- Estações_Hidrometeorológicas/MapServer
- Evap_Liq_Reservatorio_2019_MIL1/MapServer
- Evap_microbacias_2019_MIL1/MapServer
- GEOFT_UNIDADE_FEDERACAO_2013/MapServer
- Hidrografia_MIL1/MapServer
- Hidrografia/MapServer
- IEP_MIL1/MapServer
- Mapa_CODIH/MapServer
- MapaInterativoSNISB_MIL1/FeatureServer
- MapaInterativoSNISB_MIL1/MapServer
- NotasConsistencia/MapServer
- Objetivos_RHNR/MapServer
- Obrigatoriedade_Automonitoramento_DW_v4/FeatureServer
- Obrigatoriedade_Automonitoramento_DW_v4/MapServer
- Operacao2022_MIL1/MapServer
- Operacao2022_MIL2/Ma

d:\Codes\GitHub\Personal\my_projects\open-geodata\.venv\Lib\site-packages\restapi\common_types.py:1435: UserWarning: Authentation Error for folder Utilities: 
{'error': {'code': 499, 'message': 'Token Required', 'details': []}}
  warnings.warn('Authentation Error for folder {}: {}{}'.format(f, os.linesep,  e))


In [9]:
service = ags.getService(name_or_wildcard='SP')
service

<MapService: RiosPrincipais/MapServer>

In [10]:
# Seleciona Layer no Serviço
lyr = service.layer(name_or_id='RiosPrincipais')
type(lyr)

restapi.common_types.MapServiceLayer

In [11]:
lyr_query = lyr.query(
    where='1=1',
    # Se exceed_limit=True, retorna todos os registros
    # Se exceed_limit=False, retorna apenas os primeiros 1000 registros
    exceed_limit=True,
    # ------------------------------------
    # Número de registros a serem retornados
    # Se records=None, retorna todos os registros
    # records=10,
    # ------------------------------------
    # Option to return a generator with a FeatureSet in chunks of each query group.
    # Use this to avoid memory errors when fetching many features. Defaults to False
    fetch_in_chunks=True,
)

lyr_query.geometryType

AttributeError: geometryType